# **Classification: XGBoost Classifier (XGB)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
**XGBoost** is a decision-tree-based algorithm. Much like Random Forest and Gradient Boosting, it partitions data through feature thresholds. This operational logic makes the model **invariant to the scale** of the input data. Applying standardization or normalization would result in the exact same performance but incur an unnecessary computational cost. Therefore, we will utilize the **Original Data**.

### **Computational Efficiency and Regularization**
XGBoost stands out due to its efficient implementation of parallel processing and the use of **L1 and L2 regularization**, which helps control model complexity in large datasets (**100,000 rows**). Although Boosting is inherently sequential, XGBoost optimizes tree node creation in parallel, making it significantly faster than the traditional Gradient Boosting implementation in Scikit-Learn.

## **Experiment Design**

We have designed a tournament of **3 optimization levels** focused on the identified fundamental parameters:

* **Baseline**: Strict use of the `xgboost` library's **default parameters** to establish a performance benchmark.
* **GridSearchCV**: An exhaustive search using the intervals from your worksheet: **`n_estimators`** [50, 100], **`learning_rate`** [0.05, 0.1], and **`max_depth`** [3, 5].
* **Optuna**: Bayesian optimization using the exact same intervals and limits to find the ideal combination to maximize **Recall**.

In [4]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_XGBoost")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Boosting/XGBoostClassifier/mlruns/11'), creation_time=1778159048515, experiment_id='11', last_update_time=1778159048515, lifecycle_stage='active', name='Classification_XGBoost', tags={}, workspace='default'>

In [5]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

# Categorical columns
categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_metrics(y_true, y_pred, duration):
    """Log metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_Baseline_Defaults"):
    # Default parameters as per XGBoost documentation
    xgb_base = XGBClassifier(random_state=42, eval_metric="logloss")
    
    start_time = time.time()
    xgb_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = xgb_base.predict(X_test)
    
    mlflow.log_params(xgb_base.get_params())
    mlflow.log_param("optimization", "none_default")
    log_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_GridSearch"):
    param_grid = {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    }
    
    grid = GridSearchCV(
        XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred_grid = grid.best_estimator_.predict(X_test)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    log_metrics(y_test, y_pred_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    # Using the exact same ranges as the GridSearchCV for consistency
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 5)
    }
    
    model = XGBClassifier(**params)
    # CV scoring focused on Recall
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="XGB_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=10) 
    duration = time.time() - start_time
    
    # Retrain final model with Optuna's best parameters
    best_xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric="logloss")
    best_xgb.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    log_metrics(y_test, best_xgb.predict(X_test), duration)

[I 2026-05-07 14:04:51,971] A new study created in memory with name: no-name-66f1d661-8181-4193-89ab-38c7446099ab
[I 2026-05-07 14:04:53,081] Trial 0 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 93, 'learning_rate': 0.050227369897957355, 'max_depth': 4}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-07 14:04:54,103] Trial 1 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 87, 'learning_rate': 0.09028596346002685, 'max_depth': 4}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-07 14:04:55,043] Trial 2 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 58, 'learning_rate': 0.07189622522485326, 'max_depth': 5}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-07 14:04:56,155] Trial 3 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 90, 'learning_rate': 0.071181034897125, 'max_depth': 5}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-07 14:04:57,

## Runs Summary

| Run | Optimization | n_estimators | learning_rate | max_depth | subsample | Accuracy | F1 | Recall | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| XGB_Baseline_Defaults | none_default | 100 | 0.3 | 6 | 1.0 | 0.91755 | 0.9266034629 | 0.86742 | 1.92s |
| XGB_GridSearch | GridSearchCV | 50 | 0.05 | 3 | 1.0 | 0.91990 | 0.9284757568 | 0.86650 | 8.99s |
| XGB_Optuna | optuna | 93 | 0.0502273699 | 4 | 1.0 | 0.91990 | 0.9284757568 | 0.86650 | 9.68s |

### Additional logged parameters
- `random_state = 42` where set in code
- `eval_metric = logloss`
- Parâmetros otimizados: `n_estimators`, `learning_rate`, `max_depth`
- Os valores acima foram retirados do ficheiro `runs.csv`

## Best Run Justification for Streamlit

A melhor run para o Streamlit é a **XGB_Baseline_Defaults**. Apesar de a **XGB_GridSearch** e a **XGB_Optuna** apresentarem uma ligeira melhoria em Accuracy e F1, essa vantagem é muito pequena e vem acompanhada de um Recall inferior ao baseline e de um custo de treino mais elevado.

Para um classificador médico, o Recall tem peso especial porque é a métrica mais ligada à redução de falsos negativos. Neste caso, o baseline consegue o melhor Recall da tabela e, ao mesmo tempo, é de longe a opção mais rápida.

Equilíbrio final das métricas:
- **Accuracy**: 0.91755
- **F1**: 0.92660
- **Recall**: 0.86742
- **Fit time**: 1.92s

Já a **XGB_GridSearch** e a **XGB_Optuna** ficam com Accuracy e F1 iguais entre si, mas sem ganho em Recall e com tempos de treino cerca de 5 vezes superiores ao baseline. Para uso no Streamlit, isso não compensa. Por isso, a escolha mais equilibrada é a **XGB_Baseline_Defaults**, por combinar melhor Recall com menor custo computacional e uma performance global muito próxima das restantes runs.